# SENTINEL-GNSS — Kaggle Training Notebook

**Before running:** Settings ▸ Accelerator ▸ **GPU T4 x2** (or P100)  
**Also required:** Settings ▸ **Internet on** (needed to clone from GitHub)

| What lives where | |
|---|---|
| Code + all data CSVs | GitHub (cloned in Step 1) |
| Feature windows (.npz) | Built fresh in `/kaggle/working/sentinel-gnss/` |
| Checkpoints + figures | Google Drive (Step 3) **and** Kaggle Output tab |

### Google Drive persistence (optional but recommended)
Kaggle sessions expire and `/kaggle/working/` is wiped. To keep checkpoints across sessions:
1. Create a **Google Cloud service account** (console.cloud.google.com → IAM → Service Accounts)
2. Enable the **Google Drive API** on that project
3. Download the JSON key and add it as a Kaggle secret named `GDRIVE_CREDENTIALS`
4. Create a folder in your Drive, share it with the service account email, and add the folder ID as `GDRIVE_FOLDER_ID` secret

If secrets are not configured Step 3 falls back to local-only mode — everything still works, outputs are in the Kaggle Output tab.

## Step 1 — Clone repo from GitHub
All code and processed CSVs are pulled directly from GitHub.  
**Internet must be enabled** in Kaggle notebook settings.

In [ ]:
import os; os.chdir('/kaggle/working')

In [ ]:
import os, shutil

GITHUB_REPO = 'https://github.com/Jorshuare/AI-Based-Prediction-for-GNSS-Signal-Degradation.git'
REPO_DIR    = '/kaggle/working/sentinel-gnss'

%cd /kaggle/working

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Repo already cloned — pulling latest ...')
    %cd {REPO_DIR}
    !git pull
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    print('Cloning repo ...')
    !git clone {GITHUB_REPO} {REPO_DIR}
    %cd {REPO_DIR}

print(f'\nWorking directory: {os.getcwd()}')
!git log --oneline -5

csv_path = f'{REPO_DIR}/data/labelled/sentinel_gnss_labelled.csv'
assert os.path.exists(csv_path), f'CSV not found at {csv_path}'
import pandas as pd
df = pd.read_csv(csv_path)
print(f'\nDataset: {len(df):,} rows × {len(df.columns)} columns — ready.')

## Step 2 — Install extra dependencies + verify GPU

In [ ]:
!pip install -q imbalanced-learn xgboost google-api-python-client google-auth-httplib2 google-auth-oauthlib

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {props.total_memory / 1e9:.1f} GB')
    if torch.cuda.device_count() > 1:
        print(f'GPUs     : {torch.cuda.device_count()} (T4 x2)')
else:
    raise RuntimeError(
        'NO GPU — Settings ▸ Accelerator ▸ GPU T4 x2, then re-run from Step 1.')

## Step 3 — Set up output directories + Google Drive (optional)

Creates local output dirs. If Kaggle secrets `GDRIVE_CREDENTIALS` and `GDRIVE_FOLDER_ID`
are present, also authenticates with Google Drive so checkpoints and figures are mirrored
there at the end of each step. Falls back silently to local-only if secrets are missing.

In [ ]:
import os, json, shutil

REPO_DIR = '/kaggle/working/sentinel-gnss'

OUTPUT_DIRS = [
    f'{REPO_DIR}/results/models/checkpoints',
    f'{REPO_DIR}/results/models/checkpoints_lstm_only',
    f'{REPO_DIR}/results/models/checkpoints_transformer_only',
    f'{REPO_DIR}/results/figures',
    f'{REPO_DIR}/results/baselines',
    f'{REPO_DIR}/results/metrics',
]
for d in OUTPUT_DIRS:
    os.makedirs(d, exist_ok=True)
    print(f'  ✓  {d}')

# ── Google Drive setup ────────────────────────────────────────────────────────
DRIVE_ENABLED = False
drive_service = None
GDRIVE_FOLDER_ID = None

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    creds_json  = secrets.get_secret('GDRIVE_CREDENTIALS')
    GDRIVE_FOLDER_ID = secrets.get_secret('GDRIVE_FOLDER_ID')

    import tempfile
    from google.oauth2.service_account import Credentials
    from googleapiclient.discovery import build

    creds_dict = json.loads(creds_json)
    creds = Credentials.from_service_account_info(
        creds_dict,
        scopes=['https://www.googleapis.com/auth/drive']
    )
    drive_service = build('drive', 'v3', credentials=creds)
    # Test connection
    drive_service.files().get(fileId=GDRIVE_FOLDER_ID).execute()
    DRIVE_ENABLED = True
    print('\n✅ Google Drive connected — results will be mirrored to Drive.')
    print(f'   Folder ID: {GDRIVE_FOLDER_ID}')
except Exception as e:
    print(f'\n⚠️  Google Drive not configured ({type(e).__name__}: {e})')
    print('   Running in local-only mode — outputs available in the Kaggle Output tab.')
    print('   To enable Drive: add GDRIVE_CREDENTIALS and GDRIVE_FOLDER_ID to Kaggle Secrets.')


def upload_to_drive(local_path: str, filename: str = None):
    """Upload a local file to the configured Drive folder. No-op if Drive disabled."""
    if not DRIVE_ENABLED or drive_service is None:
        return
    from googleapiclient.http import MediaFileUpload
    import mimetypes
    fname = filename or os.path.basename(local_path)
    mime  = mimetypes.guess_type(local_path)[0] or 'application/octet-stream'
    # Check if file already exists in folder (update rather than duplicate)
    q = f"name='{fname}' and '{GDRIVE_FOLDER_ID}' in parents and trashed=false"
    existing = drive_service.files().list(q=q, fields='files(id)').execute().get('files', [])
    media = MediaFileUpload(local_path, mimetype=mime, resumable=True)
    if existing:
        drive_service.files().update(fileId=existing[0]['id'], media_body=media).execute()
    else:
        meta = {'name': fname, 'parents': [GDRIVE_FOLDER_ID]}
        drive_service.files().create(body=meta, media_body=media, fields='id').execute()


def sync_folder_to_drive(local_folder: str, exts: tuple = ('.pt', '.json', '.png', '.pdf', '.md', '.txt')):
    """Upload all matching files from a local folder to Drive. No-op if Drive disabled."""
    if not DRIVE_ENABLED:
        return
    uploaded = 0
    for root, _, files in os.walk(local_folder):
        for f in files:
            if any(f.endswith(e) for e in exts):
                upload_to_drive(os.path.join(root, f))
                uploaded += 1
    if uploaded:
        print(f'  → Synced {uploaded} files to Google Drive')


# Report existing checkpoints
ckpt_dir = f'{REPO_DIR}/results/models/checkpoints'
ckpts = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pt'))
if ckpts:
    print(f'\nExisting checkpoints ({len(ckpts)}): {", ".join(ckpts)}')
else:
    print('\nNo existing checkpoints — will start fresh.')

## Step 4 — Process new datasets (Deep + Harsh) — Run 12 only

> **Standard workflow: Skip this cell.** CSVs are already committed to GitHub.

| Dataset | Expected rows | Expected DEGRADED% |
|---------|---------------|-----------------|
| HK-Deep-Urban-1  (Whampoa, 10 receivers)  | ~14,000 | ~25–35% |
| HK-Harsh-Urban-1 (Mong Kok, 10 receivers) | ~30,000 | ~35–45% |

In [ ]:
%cd /kaggle/working/sentinel-gnss
import os, pandas as pd

deep_csv  = 'data/processed/urbannav/urbannav_deep_features.csv'
harsh_csv = 'data/processed/urbannav/urbannav_harsh_features.csv'

if os.path.exists(deep_csv) and os.path.exists(harsh_csv):
    df_deep  = pd.read_csv(deep_csv)
    df_harsh = pd.read_csv(harsh_csv)
    print(f"Deep  CSV: {len(df_deep):,} rows — labels: {df_deep['label'].value_counts().to_dict()}")
    print(f"Harsh CSV: {len(df_harsh):,} rows — labels: {df_harsh['label'].value_counts().to_dict()}")
    print("\nCSVs from GitHub — proceed to Step 5.")
else:
    deep_dir  = '/kaggle/input/urbannav-deep/urbanNav_Deep'
    harsh_dir = '/kaggle/input/urbannav-harsh/urbanNav_Harsh'
    if os.path.exists(deep_dir):
        !python src/processing/process_all_datasets.py --source urbannav_deep
    else:
        print(f'[SKIP] {deep_dir} not found')
    if os.path.exists(harsh_dir):
        !python src/processing/process_all_datasets.py --source urbannav_harsh
    else:
        print(f'[SKIP] {harsh_dir} not found')
    !python src/processing/process_all_datasets.py --combine

df = pd.read_csv('data/labelled/sentinel_gnss_labelled.csv')
print(f"\nCombined: {len(df):,} rows × {len(df.columns)} columns")

## Step 5 — Build feature windows

Produces two sets of windows:
- `windows/` — SMOTE-balanced (112K train) — used by **RF/XGBoost baselines only**
- `windows_no_smote/` — natural distribution (62K train) — used by **all DL models**

**Why DL uses no-SMOTE:** Run 13 confirmed SMOTE degraded the Transformer+LSTM by 1.6 pts
(0.821 → 0.804). DL handles imbalance via focal loss + class weights [1, 2, 5] instead.  
**Why baselines use SMOTE:** RF/XGBoost have no equivalent built-in imbalance mechanism.  
**Step 9 tests both** — so we can report RF with and without SMOTE for a complete comparison.

In [ ]:
%cd /kaggle/working/sentinel-gnss

# SMOTE windows — primary training set for all models
!python -m src.models.feature_prep --force

# No-SMOTE windows — kept for reference / ablation studies
!python -m src.models.feature_prep --no_smote --force

import numpy as np
for tag, wdir in [('SMOTE (primary)', 'windows'), ('no-SMOTE (reference)', 'windows_no_smote')]:
    print(f'\nWindow shapes [{tag}]:')
    for split in ('train', 'val', 'test'):
        d = np.load(f'data/processed/{wdir}/{split}.npz')
        c = int(np.sum(d['y_5s'] == 0))
        w = int(np.sum(d['y_5s'] == 1))
        g = int(np.sum(d['y_5s'] == 2))
        print(f'  {split:5s}  X={d["X"].shape}  CLEAN={c:,}  WARNING={w:,}  DEGRADED={g:,}')

## Step 6 — Train (full Transformer + BiLSTM)

Trains on **no-SMOTE windows** with focal loss + class weights (confirmed better in Run 13).  
Checkpoints saved locally and synced to Drive after training completes.

In [ ]:
import glob, os
REPO_DIR  = '/kaggle/working/sentinel-gnss'
ckpt_dir  = f'{REPO_DIR}/results/models/checkpoints'

# ══════════════════════════════════════════════════════════════════
# FRESH_START = True  → wipes ALL checkpoints before training
#                        Use this when running everything from scratch
# FRESH_START = False → keeps existing checkpoints (resume training)
#                        Use this only if continuing a previous session
# ══════════════════════════════════════════════════════════════════
FRESH_START = True

if FRESH_START:
    wiped = []
    for ckpt_path in [
        ckpt_dir,
        f'{REPO_DIR}/results/models/checkpoints_lstm_only',
        f'{REPO_DIR}/results/models/checkpoints_transformer_only',
    ]:
        for f in glob.glob(ckpt_path + '/*.pt'):
            os.remove(f)
            wiped.append(os.path.basename(f))
    if wiped:
        print(f'Wiped {len(wiped)} checkpoint(s): {wiped}')
    else:
        print('No existing checkpoints to wipe — clean start.')
else:
    ckpts = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pt'))
    if ckpts:
        print(f'Keeping {len(ckpts)} checkpoint(s): {ckpts}')
        print('Add --resume to the training command below to continue from last checkpoint.')
    else:
        print('No existing checkpoints — starting fresh.')

print(f'\nFRESH_START = {FRESH_START}')

In [ ]:
%cd /kaggle/working/sentinel-gnss
# ══════════════════════════════════════════════════════════════════
#  SENTINEL-GNSS — Full Transformer+BiLSTM  (Run 14)
#  Data:     no-SMOTE windows (focal loss handles imbalance)
#  Architecture: TransformerEncoder(2L,8H,d=128) → BiLSTM(2L,h=256)
#  Loss:     focal_gamma=1.0, class_weights=[1.0,2.0,5.0], smoothing=0.1
#  Optimiser: AdamW, patience=50, min_epoch_for_best=15, batch=256
#  Run 13 result: no-SMOTE=0.821 MacroF1 > SMOTE=0.804 MacroF1 at +5s
# ══════════════════════════════════════════════════════════════════
!python -m src.models.train \
    --batch_size 256 \
    --window_dir data/processed/windows_no_smote

sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints')

## Step 7 — Evaluate full model
Loads `checkpoint_best.pt`, runs all 14 analyses, saves figures.

In [ ]:
%cd /kaggle/working/sentinel-gnss
!python -m src.models.evaluate \
    --tune_thresholds \
    --temperature_scaling \
    --window_dir data/processed/windows_no_smote

sync_folder_to_drive(f'{REPO_DIR}/results/figures')

## Step 8 — View all figures inline

In [ ]:
import glob
from IPython.display import Image, display

figs = sorted(glob.glob('/kaggle/working/sentinel-gnss/results/figures/*.png'))
print(f'{len(figs)} figures:')
for f in figs:
    print(f'  {f.split("/")[-1]}')
    display(Image(filename=f, width=950))

## Step 9 — Baselines (Tier 1–3)

Runs RF and XGBoost **twice** — with SMOTE windows and without — for a complete comparison:
- **With SMOTE (112K balanced):** their strongest configuration
- **Without SMOTE (62K natural + class_weight='balanced'):** equal data condition vs DL

This lets the paper honestly report both configurations and explain the SMOTE effect.

In [ ]:
%cd /kaggle/working/sentinel-gnss

print('=' * 70)
print('BASELINES — Configuration A: SMOTE windows (112K balanced)')
print('=' * 70)
!python -m src.models.baselines \
    --windows_dir data/processed/windows

import json, os
smote_results = json.load(open('results/baselines/baseline_comparison.json'))
os.rename('results/baselines/baseline_comparison.json',
          'results/baselines/baseline_comparison_smote.json')

print()
print('=' * 70)
print('BASELINES — Configuration B: No-SMOTE windows (62K natural + class weights)')
print('Equal-data comparison with the DL model')
print('=' * 70)
!python -m src.models.baselines \
    --windows_dir data/processed/windows_no_smote

nsmote_results = json.load(open('results/baselines/baseline_comparison.json'))
os.rename('results/baselines/baseline_comparison.json',
          'results/baselines/baseline_comparison_no_smote.json')

# Print combined comparison
print()
print('=' * 70)
print('SMOTE vs No-SMOTE — RF and XGBoost +5s MacroF1')
print('=' * 70)
for method in ['RandomForest', 'XGBoost']:
    s  = smote_results.get(method, {}).get('5s', {}).get('overall', {}).get('macro_f1', 0)
    ns = nsmote_results.get(method, {}).get('5s', {}).get('overall', {}).get('macro_f1', 0)
    print(f'  {method:15s}  SMOTE={s:.4f}  No-SMOTE={ns:.4f}  Δ={s-ns:+.4f}')
print(f'  DL (no-SMOTE, focal loss)                    ~0.8206  (Run 13)')
print()

# Save combined results for Step 11
combined = {'smote': smote_results, 'no_smote': nsmote_results}
with open('results/baselines/baseline_comparison_combined.json', 'w') as f:
    json.dump(combined, f, indent=2)

# Restore the smote version as default (for --include_ablations)
import shutil
shutil.copy('results/baselines/baseline_comparison_smote.json',
            'results/baselines/baseline_comparison.json')

sync_folder_to_drive(f'{REPO_DIR}/results/baselines')

## Step 10 — Ablations (Tier 4)

LSTM-only and Transformer-only trained on **no-SMOTE windows** — same as the full model.  
Identical data, loss, and hyperparameters across all three DL architectures; only the
model structure differs. This is the only fair way to isolate architectural contribution.

In [ ]:
%cd /kaggle/working/sentinel-gnss
# LSTM-only ablation — no Transformer encoder
!python -m src.models.train \
    --model_type lstm_only \
    --batch_size 256 \
    --window_dir data/processed/windows_no_smote

!python -m src.models.evaluate \
    --model_type lstm_only \
    --tune_thresholds \
    --temperature_scaling \
    --window_dir data/processed/windows_no_smote

sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints_lstm_only')

In [ ]:
%cd /kaggle/working/sentinel-gnss
# Transformer-only ablation — no LSTM (mean pooling over sequence)
!python -m src.models.train \
    --model_type transformer_only \
    --batch_size 256 \
    --window_dir data/processed/windows_no_smote

!python -m src.models.evaluate \
    --model_type transformer_only \
    --tune_thresholds \
    --temperature_scaling \
    --window_dir data/processed/windows_no_smote

sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints_transformer_only')

In [ ]:
%cd /kaggle/working/sentinel-gnss
!python -m src.models.baselines --include_ablations

## Step 10b — Reviewer-Directed Experiments

Addresses four key reviewer concerns before writing the paper:

1. **Permutation shuffle test** — Proves DL uses genuine temporal structure (Reviewer 2 & 4)
2. **Inference latency** — Proves unified multi-horizon deployment advantage over 3× separate RF models (Reviewer 2 & 3)
3. **Lead-time analysis** — Extracts median seconds of advance warning from the histogram (Reviewer 3)
4. **SMOTE distribution analysis** — Explains empirically why no-SMOTE outperforms SMOTE (Reviewer 4)

In [ ]:
"""
STEP 10b — Reviewer-Directed Experiments (E1–E7)
All results saved to results/reviewer_experiments.json. Nothing assumed.
E7 calibration recomputes temperature inline (fixes the earlier T=1.0 bug).
"""
%cd /kaggle/working/sentinel-gnss
import numpy as np, torch, json, time, os, sys
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.utils import resample
from scipy.optimize import minimize_scalar
import pandas as pd

_REPO_DIR='/kaggle/working/sentinel-gnss'; _RESULTS=f'{_REPO_DIR}/results'
_DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'upload_to_drive' not in dir():
    def upload_to_drive(*a, **k): pass
    def sync_folder_to_drive(*a, **k): pass
ALL={}

FEATURE_NAMES=['alt','baseline_sats','clock_bias','cnr_trend','cnr_variance','cycle_slips','dop_ratio','elevation_violations','fix_continuity','fix_transitions','gdop','hdop','hdop_delta','iono_delay','lat_std','lon_std','max_cnr','mean_cnr','min_cnr','multipath','num_satellites','pdop','pdop_delta','position_variance','receiver_tier','residual_mean','residual_std','sat_drop_rate','sat_mean','sat_min','sat_visibility','solution_age','solution_status','std_cnr','tropo_delay','vdop','cnr_available']
TEMPORAL=['cnr_trend','cnr_variance','fix_continuity','fix_transitions','position_variance','sat_drop_rate','sat_visibility','sat_mean','sat_min']
TEMPORAL_IDX=[FEATURE_NAMES.index(f) for f in TEMPORAL]

te=np.load('data/processed/windows_no_smote/test.npz'); tr=np.load('data/processed/windows_no_smote/train.npz')
X_test,y_test=te['X'],te['y_5s']; X_train,y_train=tr['X'],tr['y_5s']
N_TEST,T_STEPS,N_FEAT=X_test.shape
print(f'Test {X_test.shape} dist={np.bincount(y_test)} | Train {X_train.shape} dist={np.bincount(y_train)}')

sys.path.insert(0,_REPO_DIR)
from src.models.transformer_lstm import SentinelGNSS
ckpt_path=f'{_RESULTS}/models/checkpoints/checkpoint_best.pt'
ckpt=torch.load(ckpt_path,map_location=_DEVICE,weights_only=False); cfg=ckpt.get('config',{})
model=SentinelGNSS(n_features=cfg.get('n_features',N_FEAT),n_classes=cfg.get('n_classes',3),d_model=cfg.get('d_model',128),n_heads=cfg.get('n_heads',8),n_tf_layers=cfg.get('n_tf_layers',2),d_ff=cfg.get('d_ff',512),lstm_hidden=cfg.get('lstm_hidden',256),n_lstm_layers=cfg.get('n_lstm_layers',2),dropout=cfg.get('dropout',0.3)).to(_DEVICE)
model.load_state_dict(ckpt['model']); model.eval()
print(f'Model epoch={ckpt["epoch"]} best={ckpt["best_metric"]:.4f} params={sum(p.numel() for p in model.parameters()):,}')

def dl_proba(X,h='5s',bs=512):
    o=[]
    for i in range(0,len(X),bs):
        xb=torch.tensor(X[i:i+bs],dtype=torch.float32).to(_DEVICE)
        with torch.no_grad(): out=model(xb)
        o.append(torch.softmax(out[f'logits_{h}'],dim=-1).cpu().numpy())
    return np.concatenate(o)
def dl_logits(X,h='5s',bs=512):
    o=[]
    for i in range(0,len(X),bs):
        xb=torch.tensor(X[i:i+bs],dtype=torch.float32).to(_DEVICE)
        with torch.no_grad(): out=model(xb)
        o.append(out[f'logits_{h}'].cpu().numpy())
    return np.concatenate(o)
def dl_pred(X,h='5s'): return dl_proba(X,h).argmax(1)
def macro(yt,yp): return f1_score(yt,yp,average='macro',zero_division=0)
def pcls(yt,yp): return f1_score(yt,yp,average=None,labels=[0,1,2],zero_division=0)
def train_rf(Xf,y,n=200): r=RandomForestClassifier(n,class_weight='balanced',n_jobs=-1,random_state=42); r.fit(Xf,y); return r

Xtrf=X_train.reshape(len(X_train),-1); Xtef=X_test.reshape(N_TEST,-1)
ydl=dl_pred(X_test); f1dl=macro(y_test,ydl); pcdl=pcls(y_test,ydl)
rf=train_rf(Xtrf,y_train); yrf=rf.predict(Xtef); f1rf=macro(y_test,yrf); pcrf=pcls(y_test,yrf)
print(f'DL ref MacroF1={f1dl:.4f} | RF ref MacroF1={f1rf:.4f}')
SEP='-'*64

print(f'\n{SEP}\nE1 - PERMUTATION SHUFFLE\n{SEP}')
np.random.seed(42); Xsh=X_test.copy()
for i in range(N_TEST): Xsh[i]=Xsh[i][np.random.permutation(T_STEPS)]
f1dlsh=macro(y_test,dl_pred(Xsh)); f1rfsh=macro(y_test,rf.predict(Xsh.reshape(N_TEST,-1)))
ALL['E1_permutation']={'dl_original':round(f1dl,4),'dl_shuffled':round(f1dlsh,4),'dl_drop':round(f1dl-f1dlsh,4),'rf_original':round(f1rf,4),'rf_shuffled':round(f1rfsh,4),'rf_drop':round(f1rf-f1rfsh,4)}
print(f'  DL drop={f1dl-f1dlsh:+.4f}  RF drop={f1rf-f1rfsh:+.4f}')

print(f'\n{SEP}\nE2 - TEMPORAL FEATURE ABLATION (RF)\n{SEP}')
nt=[i for i in range(N_FEAT) if i not in TEMPORAL_IDX]
rfnt=train_rf(X_train[:,:,nt].reshape(len(X_train),-1),y_train)
f1nt=macro(y_test,rfnt.predict(X_test[:,:,nt].reshape(N_TEST,-1)))
ALL['E2_temporal_ablation']={'rf_all':round(f1rf,4),'rf_no_temp':round(f1nt,4),'drop':round(f1rf-f1nt,4),'dl_ref':round(f1dl,4),'removed_features':TEMPORAL}
print(f'  RF all={f1rf:.4f}  RF no-temporal={f1nt:.4f}  d={f1rf-f1nt:+.4f}')

print(f'\n{SEP}\nE3 - PER-CLASS BOOTSTRAP CIs (1000x)\n{SEP}')
boot={'CLEAN':[],'WARNING':[],'DEGRADED':[],'MacroF1':[]}; np.random.seed(42)
for _ in range(1000):
    idx=resample(np.arange(N_TEST)); pc=pcls(y_test[idx],ydl[idx])
    for j,c in enumerate(['CLEAN','WARNING','DEGRADED']): boot[c].append(pc[j])
    boot['MacroF1'].append(macro(y_test[idx],ydl[idx]))
ALL['E3_per_class_bootstrap_ci']={}
for c in ['CLEAN','WARNING','DEGRADED','MacroF1']:
    a=np.array(boot[c]); ALL['E3_per_class_bootstrap_ci'][c]={'mean':round(a.mean(),4),'ci_low':round(np.percentile(a,2.5),4),'ci_high':round(np.percentile(a,97.5),4)}
    print(f'  {c:10s} {a.mean():.3f} [{np.percentile(a,2.5):.3f}, {np.percentile(a,97.5):.3f}]')

print(f'\n{SEP}\nE4 - INFERENCE LATENCY\n{SEP}')
BL,NT=256,100; xb=torch.tensor(X_test[:BL],dtype=torch.float32).to(_DEVICE); xf=X_test[:BL].reshape(BL,-1)
with torch.no_grad():
    for _ in range(5): _=model(xb)
if _DEVICE.type=='cuda': torch.cuda.synchronize()
t0=time.perf_counter()
with torch.no_grad():
    for _ in range(NT): _=model(xb)
if _DEVICE.type=='cuda': torch.cuda.synchronize()
dlms=(time.perf_counter()-t0)*1000/(NT*BL)
sm=np.load('data/processed/windows/train.npz'); Xsm=sm['X'].reshape(len(sm['X']),-1)
rf5=RandomForestClassifier(100,class_weight='balanced',n_jobs=-1,random_state=42).fit(Xsm,sm['y_5s'])
rf15=RandomForestClassifier(100,class_weight='balanced',n_jobs=-1,random_state=42).fit(Xsm,sm['y_15s'])
rf30=RandomForestClassifier(100,class_weight='balanced',n_jobs=-1,random_state=42).fit(Xsm,sm['y_30s'])
t0=time.perf_counter()
for _ in range(NT): rf5.predict(xf); rf15.predict(xf); rf30.predict(xf)
rfms=(time.perf_counter()-t0)*1000/(NT*BL)
mb=Path(ckpt_path).stat().st_size/1e6
ALL['E4_latency']={'dl_ms':round(dlms,5),'rf_ms':round(rfms,5),'speedup':round(rfms/dlms,2),'ckpt_mb':round(mb,2),'device':str(_DEVICE)}
print(f'  DL {dlms:.4f} ms  RF {rfms:.4f} ms  speedup {rfms/dlms:.1f}x  ckpt {mb:.1f}MB')

print(f'\n{SEP}\nE5 - SMOTE DISTRIBUTION (KL)\n{SEP}')
def dist(y): c=np.bincount(y,minlength=3); return c/c.sum()
def kl(p,q,e=1e-9): return float(np.sum(p*np.log((p+e)/(q+e))))
dsm=dist(np.load('data/processed/windows/train.npz')['y_5s']); dns=dist(tr['y_5s']); dte=dist(y_test)
klsm=kl(dte,dsm); klns=kl(dte,dns)
ALL['E5_smote_distribution']={'smote_train':dict(zip('CWD',dsm.round(4).tolist())),'nsmote_train':dict(zip('CWD',dns.round(4).tolist())),'test':dict(zip('CWD',dte.round(4).tolist())),'kl_smote':round(klsm,6),'kl_nsmote':round(klns,6),'closer_to_test':'SMOTE' if klsm<klns else 'no-SMOTE'}
print(f'  KL(test||SMOTE)={klsm:.5f}  KL(test||no-SMOTE)={klns:.5f}')

print(f'\n{SEP}\nE6 - CROSS-CITY TOKYO SHINJUKU\n{SEP}')
tok='data/processed/tokyo/tokyo_shinjuku_features.csv'; scl='data/processed/scaler.pkl'
if os.path.exists(tok) and os.path.exists(scl):
    import pickle
    dft=pd.read_csv(tok,low_memory=False)
    with open(scl,'rb') as fh: scaler=pickle.load(fh)
    for c in FEATURE_NAMES:
        if c not in dft.columns: dft[c]=0.0
    Xs=scaler.transform(dft[FEATURE_NAMES].fillna(0).values.astype(np.float32)); yt=dft['label'].values.astype(int)
    wx,wy=[],[]
    for i in range(T_STEPS-1,len(Xs)): wx.append(Xs[i-T_STEPS+1:i+1]); wy.append(yt[i])
    Xtw=np.array(wx,dtype=np.float32); ytw=np.array(wy,dtype=int)
    supp={'CLEAN':int(np.sum(ytw==0)),'WARNING':int(np.sum(ytw==1)),'DEGRADED':int(np.sum(ytw==2))}
    f1dlt=macro(ytw,dl_pred(Xtw)); pcdlt=pcls(ytw,dl_pred(Xtw))
    rf6=train_rf(Xtrf,y_train); f1rft=macro(ytw,rf6.predict(Xtw.reshape(len(Xtw),-1))); pcrft=pcls(ytw,rf6.predict(Xtw.reshape(len(Xtw),-1)))
    ALL['E6_cross_city_tokyo']={'n_windows':len(Xtw),'support':supp,'dl_macro_f1':round(f1dlt,4),'rf_macro_f1':round(f1rft,4),'dl_per_class':dict(zip('CWD',pcdlt.round(4).tolist())),'rf_per_class':dict(zip('CWD',pcrft.round(4).tolist())),'beijing_dl':round(f1dl,4),'beijing_rf':round(f1rf,4),'dl_gap':round(f1dlt-f1dl,4),'rf_gap':round(f1rft-f1rf,4)}
    print(f'  support={supp}')
    print(f'  DL Tokyo={f1dlt:.4f} (DEG {pcdlt[2]:.3f})  RF Tokyo={f1rft:.4f} (DEG {pcrft[2]:.3f})')
    print(f'  gap: DL {f1dlt-f1dl:+.4f}  RF {f1rft-f1rf:+.4f}')
else:
    ALL['E6_cross_city_tokyo']={'status':'skipped'}; print('  SKIPPED (file missing)')

print(f'\n{SEP}\nE7 - CALIBRATION (ECE), temperature recomputed on val\n{SEP}')
def ece_fn(yt,probs,nb=10):
    n=len(yt); conf=probs.max(1); ok=(probs.argmax(1)==yt).astype(float); v=0.0
    for lo,hi in zip(np.linspace(0,1,nb+1)[:-1],np.linspace(0,1,nb+1)[1:]):
        m=(conf>=lo)&(conf<hi)
        if m.sum(): v+=m.sum()/n*abs(conf[m].mean()-ok[m].mean())
    return v
vd=np.load('data/processed/windows_no_smote/val.npz'); Xv,yv=vd['X'],vd['y_5s']
vl=dl_logits(Xv,'5s')
def nll(T):
    z=vl/T; z=z-z.max(1,keepdims=True); p=np.exp(z); p=p/p.sum(1,keepdims=True)
    return float(-np.log(np.clip(p[np.arange(len(yv)),yv],1e-10,1)).mean())
Topt=float(minimize_scalar(nll,bounds=(0.05,5.0),method='bounded').x)
raw=dl_proba(X_test,'5s'); ece_raw=ece_fn(y_test,raw)
tl=dl_logits(X_test,'5s')/Topt; tl=tl-tl.max(1,keepdims=True); sp=np.exp(tl); sp=sp/sp.sum(1,keepdims=True)
ece_sc=ece_fn(y_test,sp)
ALL['E7_calibration']={'temperature':round(Topt,4),'ece_raw':round(ece_raw,6),'ece_scaled':round(ece_sc,6),'improvement':round(ece_raw-ece_sc,6),'well_calibrated':bool(ece_sc<0.05)}
print(f'  T={Topt:.4f}  ECE raw={ece_raw:.5f}  ECE scaled={ece_sc:.5f}  {"OK" if ece_sc<0.05 else "needs work"}')

ALL['_metadata']={'timestamp':__import__('datetime').datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC'),'dl_epoch':int(ckpt['epoch']),'n_test':N_TEST,'dl_ref_macroF1':round(f1dl,4),'rf_ref_macroF1':round(f1rf,4)}
exp=f'{_RESULTS}/reviewer_experiments.json'
json.dump(ALL,open(exp,'w'),indent=2,default=str)
print(f'\n✅ saved {exp}')
upload_to_drive(exp,'reviewer_experiments.json')

## Step 11 — Generate Run Summary Document

Collects all metrics from JSON files produced by Steps 6–10 and writes a single
human-readable summary document (`RUN_SUMMARY.md`) plus a machine-readable JSON
(`RUN_SUMMARY.json`).  Both are saved to:
- `/kaggle/working/sentinel-gnss/results/` (Kaggle Output tab)
- Google Drive (if configured in Step 3)

In [ ]:
import json, os, glob
from datetime import datetime, timezone

REPO_DIR  = '/kaggle/working/sentinel-gnss'
FIGS_DIR  = f'{REPO_DIR}/results/figures'
BASE_DIR  = f'{REPO_DIR}/results/baselines'
OUT_DIR   = f'{REPO_DIR}/results'

RUN_TS = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')

def load_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None

# ── Load metrics ───────────────────────────────────────────────────────────────
metrics_full   = load_json(f'{FIGS_DIR}/metrics_test.json')
metrics_lstm   = load_json(f'{FIGS_DIR}/metrics_test_lstm_only.json')
metrics_trans  = load_json(f'{FIGS_DIR}/metrics_test_transformer_only.json')
baselines_smote  = load_json(f'{BASE_DIR}/baseline_comparison_smote.json') \
                   or load_json(f'{BASE_DIR}/baseline_comparison.json')
baselines_nsmote = load_json(f'{BASE_DIR}/baseline_comparison_no_smote.json')
thresholds     = load_json(f'{FIGS_DIR}/tuned_thresholds.json')

hist_full  = load_json(f'{REPO_DIR}/results/models/checkpoints/training_history.json')
hist_lstm  = load_json(f'{REPO_DIR}/results/models/checkpoints_lstm_only/training_history.json')
hist_trans = load_json(f'{REPO_DIR}/results/models/checkpoints_transformer_only/training_history.json')

def parse_history(h):
    """Derive training summary from history lists saved by train.py."""
    if not h:
        return None
    train_loss = h.get('train_loss', [])
    # The combined stop metric is the average of stop_f1_5s/15s/30s per epoch
    s5  = h.get('stop_f1_5s',  [0])
    s15 = h.get('stop_f1_15s', [0])
    s30 = h.get('stop_f1_30s', [0])
    combined = [(a + b + c) / 3 for a, b, c in zip(s5, s15, s30)]
    best_idx = combined.index(max(combined)) if combined else 0
    return {
        'total_epochs': len(train_loss),
        'best_epoch':   best_idx + 1,
        'best_val_f1':  round(max(combined), 4) if combined else '?',
        'best_val_f1_5s': round(s5[best_idx], 4) if s5 else '?',
    }

hs_full  = parse_history(hist_full)
hs_lstm  = parse_history(hist_lstm)
hs_trans = parse_history(hist_trans)

# ── Helpers ────────────────────────────────────────────────────────────────────
def fmt_model_metrics(m, name):
    """Format DL model metrics. JSON keys are '5s','15s','30s' (no + prefix)."""
    if not m:
        return f'### {name}\n  metrics not found\n'
    lines = [f'### {name}']
    lines.append('| Horizon | Accuracy | MacroF1 | WtF1 | κ | MCC |')
    lines.append('|---|---|---|---|---|---|')
    for h_key, h_label in [('5s', '+5s'), ('15s', '+15s'), ('30s', '+30s')]:
        if h_key not in m:
            continue
        r = m[h_key]
        lines.append(
            f"| {h_label} | {r.get('accuracy',0):.4f} | **{r.get('macro_f1',0):.4f}** | "
            f"{r.get('weighted_f1',0):.4f} | {r.get('kappa',0):.4f} | {r.get('mcc',0):.4f} |"
        )
    pc = m.get('5s', {}).get('per_class', {})
    if pc:
        lines.append('')
        lines.append('**Per-class @ +5s:**')
        lines.append('| Class | Precision | Recall | F1 | Support |')
        lines.append('|---|---|---|---|---|')
        for cls, v in pc.items():
            lines.append(f"| {cls} | {v.get('precision',0):.3f} | {v.get('recall',0):.3f} | {v.get('f1',0):.3f} | {int(v.get('support',0))} |")
    ci = m.get('bootstrap_ci', {})
    if ci:
        lines.append('')
        lines.append('**Bootstrap 95% CIs:**')
        for h_key, h_label in [('5s', '+5s'), ('15s', '+15s'), ('30s', '+30s')]:
            c = ci.get(h_key, {})
            mf1 = c.get('macro_f1', ['?','?'])
            mcc = c.get('mcc', ['?','?'])
            try:
                lines.append(f'- {h_label}: MacroF1=[{mf1[0]:.3f}, {mf1[1]:.3f}]  MCC=[{mcc[0]:.3f}, {mcc[1]:.3f}]')
            except Exception:
                pass
    return '\n'.join(lines) + '\n'

def baseline_f1(b, method, horizon):
    """Extract MacroF1 for a baseline method at a given horizon."""
    try:
        return b[method][horizon]['overall']['macro_f1']
    except Exception:
        return None

def baseline_mcc(b, method, horizon):
    try:
        return b[method][horizon]['overall']['mcc']
    except Exception:
        return None

def dl_f1(m, horizon):
    """Extract MacroF1 for a DL model at a given horizon (keys without +)."""
    try:
        return m[horizon]['macro_f1']
    except Exception:
        return None

def dl_mcc(m, horizon):
    try:
        return m[horizon]['mcc']
    except Exception:
        return None

# ── Build Markdown ─────────────────────────────────────────────────────────────
md = [
    '# SENTINEL-GNSS — Run Summary',
    f'**Generated:** {RUN_TS}  ',
    f'**Repo:** https://github.com/Jorshuare/AI-Based-Prediction-for-GNSS-Signal-Degradation  ',
    '',
    '## Training Summary (best checkpoint by combined stop-F1 across 3 horizons)',
]

for label, hs in [('Transformer + LSTM (full)', hs_full),
                   ('LSTM-only ablation',         hs_lstm),
                   ('Transformer-only ablation',  hs_trans)]:
    if hs:
        md.append(f'- **{label}**: best epoch {hs["best_epoch"]}/{hs["total_epochs"]},'
                  f' best combined stop-MacroF1 = {hs["best_val_f1"]}')
    else:
        md.append(f'- **{label}**: history not found')

md += ['', '## Complete Comparison Table — Test Set MacroF1', '']
md.append('| Method | Data | +5s MacroF1 | +15s MacroF1 | +30s MacroF1 | +5s MCC |')
md.append('|---|---|---|---|---|---|')
md.append('| MajorityClass | — | 0.2016 | 0.0720 | 0.0716 | 0.000 |')
md.append('| CNR Threshold | — | 0.0735 | 0.0720 | 0.0716 | 0.000 |')

for label, b, tag in [('RandomForest (SMOTE)', baselines_smote, 'SMOTE 112K'),
                        ('XGBoost (SMOTE)',      baselines_smote, 'SMOTE 112K'),
                        ('RandomForest (no-SMOTE)', baselines_nsmote, 'no-SMOTE 62K'),
                        ('XGBoost (no-SMOTE)',      baselines_nsmote, 'no-SMOTE 62K')]:
    method = label.split(' (')[0]
    if b:
        f5  = baseline_f1(b, method, '5s')
        f15 = baseline_f1(b, method, '15s')
        f30 = baseline_f1(b, method, '30s')
        m5  = baseline_mcc(b, method, '5s')
        vals = [f'{v:.4f}' if v is not None else '—' for v in [f5, f15, f30, m5]]
        md.append(f'| {label} | {tag} | {vals[0]} | {vals[1]} | {vals[2]} | {vals[3]} |')
    else:
        md.append(f'| {label} | {tag} | — | — | — | — |')

for label, m in [('Transformer-only', metrics_trans),
                  ('LSTM-only',         metrics_lstm),
                  ('**Transformer + LSTM (SENTINEL-GNSS)**', metrics_full)]:
    if m:
        f5  = dl_f1(m, '5s');  f15 = dl_f1(m, '15s');  f30 = dl_f1(m, '30s')
        m5  = dl_mcc(m, '5s')
        vals = [f'{v:.4f}' if v is not None else '—' for v in [f5, f15, f30, m5]]
        md.append(f'| {label} | no-SMOTE + focal | {vals[0]} | {vals[1]} | {vals[2]} | {vals[3]} |')
    else:
        md.append(f'| {label} | no-SMOTE + focal | — | — | — | — |')

md += ['', '## SMOTE Effect on Classical ML']
for method in ['RandomForest', 'XGBoost']:
    s  = baseline_f1(baselines_smote,  method, '5s') if baselines_smote  else None
    ns = baseline_f1(baselines_nsmote, method, '5s') if baselines_nsmote else None
    if s is not None and ns is not None:
        md.append(f'- **{method}**: SMOTE={s:.4f}  no-SMOTE={ns:.4f}  Δ={s-ns:+.4f}')

md += ['', '## Per-Model Detail']
md.append(fmt_model_metrics(metrics_full,  'Transformer + LSTM (full model)'))
md.append(fmt_model_metrics(metrics_lstm,  'LSTM-only ablation'))
md.append(fmt_model_metrics(metrics_trans, 'Transformer-only ablation'))

md += ['', '## Data & Split Summary']
try:
    import numpy as np
    for tag, wdir in [('SMOTE', 'windows'), ('no-SMOTE', 'windows_no_smote')]:
        d = np.load(f'{REPO_DIR}/data/processed/{wdir}/train.npz')
        c = int(np.sum(d['y_5s']==0)); w = int(np.sum(d['y_5s']==1)); g = int(np.sum(d['y_5s']==2))
        md.append(f'- **Train ({tag}):** {d["X"].shape[0]:,} — CLEAN={c:,} WARNING={w:,} DEGRADED={g:,}')
    for split_tag, wdir, split in [('Val', 'windows', 'val'), ('Test', 'windows', 'test')]:
        ds = np.load(f'{REPO_DIR}/data/processed/{wdir}/{split}.npz')
        c = int(np.sum(ds['y_5s']==0)); w = int(np.sum(ds['y_5s']==1)); g = int(np.sum(ds['y_5s']==2))
        md.append(f'- **{split_tag}:** {ds["X"].shape[0]:,} — CLEAN={c:,} WARNING={w:,} DEGRADED={g:,}')
except Exception as e:
    md.append(f'  (window stats unavailable: {e})')

md += ['', '## Figures Generated']
for f in sorted(glob.glob(f'{FIGS_DIR}/*.png')):
    md.append(f'- {os.path.basename(f)}')

md_text = '\n'.join(md)

# ── Save ───────────────────────────────────────────────────────────────────────
summary_json = {
    'run_timestamp': RUN_TS,
    'training_summary': {'full': hs_full, 'lstm_only': hs_lstm, 'transformer_only': hs_trans},
    'test_metrics': {
        'full_model':         metrics_full,
        'lstm_only':          metrics_lstm,
        'transformer_only':   metrics_trans,
    },
    'baselines_smote':    baselines_smote,
    'baselines_no_smote': baselines_nsmote,
    'tuned_thresholds':   thresholds,
}

md_path   = f'{OUT_DIR}/RUN_SUMMARY.md'
json_path = f'{OUT_DIR}/RUN_SUMMARY.json'

with open(md_path, 'w') as f:
    f.write(md_text)
with open(json_path, 'w') as f:
    json.dump(summary_json, f, indent=2, default=str)

print(f'✅ {md_path}')
print(f'✅ {json_path}')
print()
print(md_text)

upload_to_drive(md_path,   'RUN_SUMMARY.md')
upload_to_drive(json_path, 'RUN_SUMMARY.json')
sync_folder_to_drive(f'{REPO_DIR}/results/figures')
sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints')
sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints_lstm_only')
sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints_transformer_only')

if DRIVE_ENABLED:
    print('\n✅ All results synced to Google Drive.')
else:
    print('\n📁 Download RUN_SUMMARY.md and RUN_SUMMARY.json from the Output tab.')

## Output

```
results/
├── RUN_SUMMARY.md             ← human-readable metrics summary  ← DOWNLOAD THIS
├── RUN_SUMMARY.json           ← machine-readable full metrics   ← DOWNLOAD THIS
├── models/
│   ├── checkpoints/           ← full model .pt files
│   ├── checkpoints_lstm_only/
│   └── checkpoints_transformer_only/
├── figures/                   ← all evaluation plots (.png/.pdf)
└── baselines/                 ← baseline_comparison.json
```

**If Drive is configured:** everything is automatically uploaded after each step.  
**If Drive is not configured:** download `RUN_SUMMARY.md` and `RUN_SUMMARY.json`  
from the Output tab — paste `RUN_SUMMARY.json` directly into the analysis session.